# 01 — Limpieza y segmentación del corpus

Este notebook limpia los textos crudos extraídos del sitio web y los PDFs,
elimina ruido de navegación y segmenta por párrafos y secciones.


In [ ]:
import re
import pandas as pd
from pathlib import Path

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(exist_ok=True)

In [ ]:
PATRONES_RUIDO = [
    r'LogoDP-sin-bandera[\s\S]*?Youtube\n',
    r'Links\nInicio[\s\S]*?Únete al Movimiento\n',
    r'© \d{4} Defensores.*?\n',
    r'Firmes por la Patria\.\s*$',
    r'Facebook-f Instagram Twitter Tiktok Youtube',
]

def limpiar_ruido(texto):
    for patron in PATRONES_RUIDO:
        texto = re.sub(patron, '', texto, flags=re.MULTILINE | re.DOTALL)
    texto = re.sub(r'\n{3,}', '\n\n', texto)
    return texto.strip()

def segmentar_parrafos(texto, fuente_id):
    parrafos = [p.strip() for p in texto.split('\n\n') if len(p.strip()) > 50]
    registros = []
    seccion_actual = 'INTRODUCCIÓN'
    for i, p in enumerate(parrafos):
        if p.isupper() and len(p) < 200:
            seccion_actual = p
            continue
        registros.append({
            'id': f'{fuente_id}_{i:04d}',
            'fuente': fuente_id,
            'seccion': seccion_actual,
            'parrafo': p,
            'n_palabras': len(p.split()),
        })
    return registros

In [ ]:
# Procesar archivo principal
# Ajustar path según nombre real del archivo
archivo = RAW_DIR / 'pilares_web_raw.txt'

if archivo.exists():
    texto = archivo.read_text(encoding='utf-8')
    texto_limpio = limpiar_ruido(texto)
    registros = segmentar_parrafos(texto_limpio, 'F02')
    df = pd.DataFrame(registros)
    df.to_csv(PROCESSED_DIR / 'corpus_limpio.csv', index=False, encoding='utf-8')
    print(f'Registros generados: {len(df)}')
    print(df.head())
else:
    print('Archivo no encontrado. Coloca el TXT en data/raw/')